# 04 · The data pipeline: from a text file to clean arrays

In the homework, the first three sub-tasks (A, B, C) had nothing to do with neural networks — and everything after them depended on getting them right. Expect the same in the coding test: you receive raw text and must produce correct `float32` arrays **without pandas-style convenience and without loops over samples** (parsing text is the accepted exception).

---
## Theory in 60 seconds

The order of the steps matters. A safe order:

1. **Parse** text → numeric array, missing values become `np.nan`, categories become integer indices.
2. **Shuffle** the rows (files are often sorted by something), then **split** into train / validation / test.
3. **Fit** the preprocessing on the **training split only** (medians for imputation, mean/std for scaling) …
4. … and **apply** those same numbers to all three splits.
5. Separate inputs `X` from labels `y`. Never leave a label column (or something computed from it) among the inputs.

Why step 3 uses only the training split: the validation and test sets stand in for *future, unseen* data. Anything computed from them and used in training is **leakage** — the reported performance becomes optimistic.

Ways to treat a missing value (lecture, week 8): replace by a constant / mean / median; add a **flag column** that says "this value was missing"; for categories add an extra "missing" category; or drop the row/column. Flag + neutral fill is the most informative: the network can learn that "missing" itself means something.

**Categorical** variables with no natural order (district, colour) → **one-hot**: one 0/1 column per category. An integer code `0,1,2,…` would wrongly tell the network that category 2 is "twice" category 1.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np
import torch
from dnd_utils import check, DATA, reference_04

REF = reference_04()                                   # expected results, used by the check cells only
content = (DATA / "houses.csv").read_text(encoding="utf-8")
print(len(content), "characters")
print(content[:330])

36687 characters
area_m2,rooms,age_years,dist_center_km,district,garden,price_keur,sold_fast
71.6,3,57,2.24,Buda,0,233.1,0
69.3,3,76,2.00,Buda,0,214.9,0
36.9,2,,4.91,Buda,0,137.5,0
72.5,2,45,3.55,Buda,1,249.0,1
95.8,2,17,0.30,Buda,0,328.6,1
48.8,2,32,4.50,Buda,0,158.6,0
90.7,4,45,3.71,Buda,0,261.8,0
25.0,1,38,0.30,Buda,0,172.6,1
98.1,3,50,5.53,B


`houses.csv`: one house per line, comma separated. `price_keur` (price in thousand EUR) and `sold_fast` (0/1) are **labels**; everything else is an input. Missing values appear as nothing between two commas.

---
## Exercise 4.1 · Parse the string

Turn `content` into a NumPy array `raw` of shape `(1200, 8)`, dtype `float32`, columns in file order:

- numeric fields → their value, empty field → `np.nan`
- `district` (column 4) → its index in `DISTRICTS`, empty → `np.nan`

A Python loop / comprehension over lines is fine here (text cannot be vectorised).

<details><summary>Hint</summary>

`content.split("\n")` gives the lines (watch out for an empty last line). `line.split(",")` gives the fields. Handle the district field separately from the numeric ones.
</details>

In [2]:
DISTRICTS = ["Buda", "Pest", "Obuda", "Ujpest", "Csepel"]

def parse_field(col_idx, text):
    if text == "":
        return np.nan
    if col_idx == 4:
        return float(DISTRICTS.index(text))
    return float(text)

lines = content.split("\n")
rows = [line.split(",") for line in lines[1:] if line]            # skip the header and the empty last line
raw = np.array([[parse_field(j, v) for j, v in enumerate(row)] for row in rows], dtype=np.float32)
print(raw.shape, raw.dtype)
print(raw[:3])

(1200, 8) float32
[[ 71.6    3.    57.     2.24   0.     0.   233.1    0.  ]
 [ 69.3    3.    76.     2.     0.     0.   214.9    0.  ]
 [ 36.9    2.      nan   4.91   0.     0.   137.5    0.  ]]


In [3]:
check("shape (1200, 8)", lambda: raw.shape == (1200, 8))
check("dtype float32", lambda: raw.dtype == np.float32)
check("values (NaNs in the same places)", lambda: np.allclose(raw, REF["raw"], equal_nan=True))

✅ shape (1200, 8)
✅ dtype float32
✅ values (NaNs in the same places)


True

---
## Exercise 4.2 · Look at the missing values

Vectorised:

- `nan_per_col`: number of NaNs in each column, shape `(8,)`
- `n_rows_with_nan`: number of rows that contain at least one NaN
- `cols_with_nan`: the indices of the columns that contain at least one NaN (a 1-D integer array)

In [4]:
nan_mask = np.isnan(raw)                               # (1200, 8) bool
nan_per_col = nan_mask.sum(axis=0)                     # collapse rows -> per column
n_rows_with_nan = int(nan_mask.any(axis=1).sum())      # any() along the columns -> per row
cols_with_nan = np.where(nan_mask.any(axis=0))[0]      # one-argument where returns a tuple of index arrays
print(nan_per_col, n_rows_with_nan, cols_with_nan)

[ 0  0 91 49 60  0  0  0] 186 [2 3 4]


In [5]:
check("nan_per_col", lambda: (np.asarray(nan_per_col) == np.isnan(REF["raw"]).sum(axis=0)).all())
check("n_rows_with_nan", lambda: n_rows_with_nan == int(np.isnan(REF["raw"]).any(axis=1).sum()))
check("cols_with_nan == [2, 3, 4]", lambda: list(cols_with_nan) == [2, 3, 4])

✅ nan_per_col
✅ n_rows_with_nan
✅ cols_with_nan == [2, 3, 4]


True

Dropping every row with a NaN would throw away roughly 16 % of the data. And why can we not just leave the NaNs in? Run the next cell.

In [6]:
layer = torch.nn.Linear(8, 1)
batch = torch.tensor(raw[:64])
out = layer(batch)
print("NaNs in the batch:", torch.isnan(batch).sum().item(), "of", batch.numel(), "values")
print("NaNs in the output:", torch.isnan(out).sum().item(), "of", out.numel())
print("loss:", out.mean().item())

NaNs in the batch: 8 of 512 values
NaNs in the output: 8 of 64
loss: nan


A single NaN anywhere in a row poisons that row's output, the mean over the batch, the loss, every gradient and — after one `step()` — **every weight of the network**. A loss of `nan` in the first epoch almost always means: NaNs (or infinities) in the data.

---
## Exercise 4.3 · Flag columns

For each column that can contain NaN (`2` age, `3` distance, `4` district), append a new column that is `1.0` where the value is **present** and `0.0` where it is missing. Result: `data`, shape `(1200, 11)`; the three flags are columns 8, 9, 10, in that order. Do **not** fill the NaNs yet.

In [7]:
valid = (~np.isnan(raw[:, [2, 3, 4]])).astype(np.float32)        # (1200, 3): 1 = value present
data = np.concatenate([raw, valid], axis=1)                       # new columns go on the right
print(data.shape)

(1200, 11)


In [8]:
check("shape (1200, 11), float32", lambda: data.shape == (1200, 11) and data.dtype == np.float32)
check("flag columns", lambda: np.array_equal(data[:, 8:], REF["data"][:, 8:]))
check("original columns untouched (NaNs still there)", lambda: np.allclose(data[:, :8], REF["raw"], equal_nan=True))

✅ shape (1200, 11), float32
✅ flag columns
✅ original columns untouched (NaNs still there)


True

---
## Exercise 4.4 · Is the file sorted? (why we shuffle)

Compute the mean price of the **first 60 %** of the rows and of the **last 20 %** of the rows of `data`, in file order.

In [9]:
n = data.shape[0]
mean_price_first60 = data[:int(n * 0.6), 6].mean()
mean_price_last20 = data[int(n * 0.8):, 6].mean()
print(mean_price_first60, mean_price_last20)

240.82668 122.70083


In [10]:
check("first 60 %", lambda: abs(mean_price_first60 - REF["data"][:720, 6].mean()) < 0.01)
check("last 20 %", lambda: abs(mean_price_last20 - REF["data"][960:, 6].mean()) < 0.01)
print("districts in the last 240 rows:", np.unique(data[960:, 4][~np.isnan(data[960:, 4])]))

✅ first 60 %
✅ last 20 %
districts in the last 240 rows: [3. 4.]


The file is ordered by district. Without shuffling, the "test set" would consist almost only of the two cheapest districts — districts the model would hardly have seen during training. What would you observe if you trained and evaluated like that?

**Answer.** The training loss would look fine, but validation/test loss would be much worse than training loss from the very first epoch — not because of over-fitting but because the splits come from **different distributions**: the model is asked about districts (and a price range) it never or hardly saw. All conclusions drawn from such curves (when to stop, how good the model is) would be meaningless. Shuffling makes the three splits statistically alike.

---
## Exercise 4.5 · Shuffle and split

- Create the permutation with **exactly** `np.random.default_rng(0).permutation(n)` (so the result can be checked) and reorder the rows of `data` with it.
- Split 60 % / 20 % / 20 % (use `int(n * ratio)` for the first two sizes; the test set gets the rest).

Result: `split_train`, `split_val`, `split_test`.

In [11]:
n = data.shape[0]
perm = np.random.default_rng(0).permutation(n)       # the integers 0..n-1 in random order
shuffled = data[perm]                                # integer-array indexing on axis 0 reorders whole rows

n_train = int(n * 0.6)
n_val = int(n * 0.2)
split_train = shuffled[:n_train]
split_val = shuffled[n_train:n_train + n_val]
split_test = shuffled[n_train + n_val:]
print(split_train.shape, split_val.shape, split_test.shape)

(720, 11) (240, 11) (240, 11)


In [12]:
check("shapes (720, 11) (240, 11) (240, 11)", lambda: (split_train.shape, split_val.shape, split_test.shape) == ((720, 11), (240, 11), (240, 11)))
check("train rows", lambda: np.allclose(split_train, REF["train"], equal_nan=True))
check("val rows", lambda: np.allclose(split_val, REF["val"], equal_nan=True))
check("test rows", lambda: np.allclose(split_test, REF["test"], equal_nan=True))
print("mean price  train %.1f  val %.1f  test %.1f" % (split_train[:, 6].mean(), split_val[:, 6].mean(), split_test[:, 6].mean()))

✅ shapes (720, 11) (240, 11) (240, 11)
✅ train rows
✅ val rows
✅ test rows
mean price  train 205.3  val 190.3  test 205.2


If inputs and labels were in **two separate arrays** `X` and `y`, how would you shuffle them? What is wrong with calling `np.random.shuffle(X)` and then `np.random.shuffle(y)`?

**Answer.** Create **one** permutation and index both with it: `perm = rng.permutation(n); X = X[perm]; y = y[perm]`. Two independent shuffles destroy the pairing between each input row and its label — the labels become random with respect to the inputs, and the network can learn nothing beyond the label mean (the loss stays at baseline level). There is no error message for this bug.

---
## Exercise 4.6 · Imputation values — from the training split only

Compute `med`: the median of `age` (column 2) and of `dist` (column 3) **in `split_train`**, ignoring NaNs. Shape `(2,)`.

In [13]:
med = np.nanmedian(split_train[:, [2, 3]], axis=0)
print(med)

[46.         5.0299997]


In [14]:
check("med", lambda: np.allclose(med, REF["med"]))

✅ med


True

---
## Exercise 4.7 · Build the input matrix

Write `build_X(split, med)` that returns the **input** matrix for any split, shape `(n, 13)`, columns in this order:

| cols | content |
|---|---|
| 0–3 | `area`, `rooms`, `age`, `dist` — with NaNs in `age`/`dist` replaced by `med[0]`/`med[1]` |
| 4 | `garden` |
| 5–9 | one-hot of `district` (5 columns, order of `DISTRICTS`); a missing district is **all zeros** |
| 10–12 | the three flag columns |

The label columns (6 and 7) must not be in it. No loops.

<details><summary>Hint — one-hot with NaNs</summary>

`split[:, 4, None] == np.arange(5)` compares a `(n, 1)` column with a `(5,)` vector → `(n, 5)` booleans. `NaN == anything` is `False`, so missing districts automatically become a row of zeros.
</details>

In [15]:
def build_X(split, med):
    num = split[:, :4].copy()                                            # copy: do not modify the caller's array
    num[:, 2:4] = np.where(np.isnan(num[:, 2:4]), med, num[:, 2:4])      # (n,2) mask, (2,) medians broadcast per column
    garden = split[:, 5, None]                                           # (n, 1)
    onehot = (split[:, 4, None] == np.arange(5)[None, :]).astype(np.float32)   # (n, 5); NaN -> all False
    flags = split[:, 8:11]
    return np.concatenate([num, garden, onehot, flags], axis=1)

In [16]:
check("shape (720, 13), float32", lambda: build_X(split_train, med).shape == (720, 13) and build_X(split_train, med).dtype == np.float32)
check("no NaN left", lambda: not np.isnan(build_X(split_train, med)).any())
check("values", lambda: np.allclose(build_X(split_train, med), REF["X_train_unscaled"]))
check("does not modify its input", lambda: np.isnan(split_train).any())
check("one-hot rows sum to the district flag", lambda: np.array_equal(build_X(split_val, med)[:, 5:10].sum(axis=1), split_val[:, 10]))

✅ shape (720, 13), float32
✅ no NaN left
✅ values
✅ does not modify its input
✅ one-hot rows sum to the district flag


True

---
## Exercise 4.8 · Standardise — fit on train, apply to all

- Compute `mean` and `std` (shape `(4,)`) of the four numeric columns of the **training** input matrix.
- Produce `X_train`, `X_val`, `X_test`: numeric columns 0–3 standardised with those values, the 0/1 columns left as they are.
- Produce the regression labels `y_train`, `y_val`, `y_test` (price, column 6) with shape `(n, 1)`.

In [17]:
Xtr_unscaled = build_X(split_train, med)
mean = Xtr_unscaled[:, :4].mean(axis=0)
std = Xtr_unscaled[:, :4].std(axis=0)

def scale(X):
    X = X.copy()
    X[:, :4] = (X[:, :4] - mean) / std            # the SAME train mean/std for every split
    return X

X_train = scale(Xtr_unscaled)
X_val = scale(build_X(split_val, med))
X_test = scale(build_X(split_test, med))

y_train = split_train[:, 6, None]                 # (n, 1)
y_val = split_val[:, 6, None]
y_test = split_test[:, 6, None]

In [18]:
check("mean, std", lambda: np.allclose(mean, REF["mean"]) and np.allclose(std, REF["std"]))
for name in ("X_train", "X_val", "X_test", "y_train", "y_val", "y_test"):
    check(name, lambda: np.allclose(globals()[name], REF[name], atol=1e-5) and globals()[name].dtype == np.float32)
print("train means of numeric cols:", X_train[:, :4].mean(axis=0).round(4))
print("val   means of numeric cols:", X_val[:, :4].mean(axis=0).round(4))

✅ mean, std
✅ X_train
✅ X_val
✅ X_test
✅ y_train
✅ y_val
✅ y_test
train means of numeric cols: [-0. -0. -0.  0.]
val   means of numeric cols: [-0.1754 -0.1742  0.0561  0.0107]


The validation means are **not** exactly 0. Is that a bug? And why were the 0/1 columns not standardised?

**Answer.** Not a bug — it is the proof that you did it right. The validation set was shifted by the *training* mean; its own mean is slightly different, so the result is close to but not exactly 0. If the validation means were exactly 0 you would have used validation statistics (leakage). The 0/1 columns already live on a scale comparable to standardised features (values of size ~1), so scaling them is unnecessary; standardising them is not *wrong*, but it destroys their readable 0/1 meaning and, for rare categories, produces a few very large values.

---
## Exercise 4.9 · Find the leaks

Each snippet below prepares data. Which contain **leakage or a pairing bug**, and what exactly is wrong? (Nothing to run — reason about it.)

```python
# (A)
X = (X - X.mean(axis=0)) / X.std(axis=0)
X_train, X_test = X[:800], X[800:]

# (B)
X_train, X_test = X[:800], X[800:]
X_train = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)
X_test = (X_test - X_test.mean(axis=0)) / X_test.std(axis=0)

# (C)
perm = rng.permutation(len(X))
X = X[perm]
X_train, y_train = X[:800], y[:800]

# (D)   houses: predict price
X = data[:, [0, 1, 2, 3, 5, 7]]      # area, rooms, age, dist, garden, sold_fast
y = data[:, 6, None]
```

**Answer.** **(A)** Leakage: mean/std are computed from all rows, including the future test rows. **(B)** No information flows from test into training, but the test set is scaled with *different* numbers than the training set, so the same raw value means something different to the model at test time (and for a single new sample you could not compute a std at all) — wrong; use the training statistics. **(C)** Pairing bug: `X` was permuted but `y` was not, so labels no longer belong to their rows. **(D)** `sold_fast` is the *other label*: it is only known after the house is on the market, and it depends on the price. Using it as an input is target leakage — at prediction time it would not be available.

---
## Blank-page drill

Write the **whole pipeline as one function** from memory, in the empty cell:

```python
def prepare_houses(content, seed=0):
    ...
    return X_train, y_train, X_val, y_val, X_test, y_test      # float32 NumPy arrays; y has shape (n, 1)
```

Same recipe as above (flags, shuffle with `default_rng(seed)`, 60/20/20, train medians, one-hot, train mean/std on the four numeric columns). The check compares it with your step-by-step result. In the test you will not have a reference — so before running the check, **verify your own result the way you would there**: print shapes, dtypes, `np.isnan(...).sum()`, column means.

In [19]:
def prepare_houses(content, seed=0):
    districts = ["Buda", "Pest", "Obuda", "Ujpest", "Csepel"]
    rows = [line.split(",") for line in content.split("\n")[1:] if line]
    raw = np.array([[np.nan if v == "" else (districts.index(v) if j == 4 else float(v)) for j, v in enumerate(r)]
                    for r in rows], dtype=np.float32)
    data = np.concatenate([raw, (~np.isnan(raw[:, [2, 3, 4]])).astype(np.float32)], axis=1)

    n = len(data)
    data = data[np.random.default_rng(seed).permutation(n)]
    n_tr, n_va = int(n * 0.6), int(n * 0.2)
    splits = [data[:n_tr], data[n_tr:n_tr + n_va], data[n_tr + n_va:]]

    med = np.nanmedian(splits[0][:, [2, 3]], axis=0)

    def build(s):
        num = s[:, :4].copy()
        num[:, 2:4] = np.where(np.isnan(num[:, 2:4]), med, num[:, 2:4])
        onehot = (s[:, 4, None] == np.arange(5)).astype(np.float32)
        return np.concatenate([num, s[:, 5, None], onehot, s[:, 8:11]], axis=1)

    Xs = [build(s) for s in splits]
    mean, std = Xs[0][:, :4].mean(axis=0), Xs[0][:, :4].std(axis=0)
    out = []
    for X, s in zip(Xs, splits):
        X[:, :4] = (X[:, :4] - mean) / std
        out += [X, s[:, 6, None]]
    return tuple(out)

In [20]:
res = None
try:
    res = prepare_houses(content, 0)
    for a in res: print(a.shape, a.dtype, int(np.isnan(a).sum()))
except NotImplementedError:
    pass
for i, name in enumerate(("X_train", "y_train", "X_val", "y_val", "X_test", "y_test")):
    check(f"prepare_houses: {name}", lambda: np.allclose(res[i], REF[name], atol=1e-5) and res[i].dtype == np.float32)

(720, 13) float32 0
(720, 1) float32 0
(240, 13) float32 0
(240, 1) float32 0
(240, 13) float32 0
(240, 1) float32 0
✅ prepare_houses: X_train
✅ prepare_houses: y_train
✅ prepare_houses: X_val
✅ prepare_houses: y_val
✅ prepare_houses: X_test
✅ prepare_houses: y_test
